In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np, pandas as pd, json, time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import beta as BetaDist, binom
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

SEED = 42
DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
DS_LABEL = {'nsl_kdd_v2': 'NSL-KDD', 'unsw_nb15_v2': 'UNSW-NB15', 'cic_ids2017_v2': 'CIC-IDS2017'}
# The three datasets form an exchangeability ladder: CIC partitions are stratified from one pool, UNSW
# train and test differ in composition, NSL-KDD's test partition was built to differ from its training set.
EXCHANGEABLE = {'cic_ids2017_v2': 'exchangeable', 'unsw_nb15_v2': 'composition shift', 'nsl_kdd_v2': 'non-exchangeable by construction'}
MODELS = [f'{a}_{v}' for v in ['5class_cw', '5class_smote'] for a in ['rf', 'xgb', 'dnn']]
CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
ATTACK_CLASSES = [1, 2, 3, 4]
K, EPS, PLATT_THRESHOLD = 5, 1e-12, 30

DELTA = 0.10                 # the certificate must hold with probability at least 1 - DELTA
SPLIT = 0.5                  # share of DELTA spent on the calibration stage, the rest on the future stage
BUDGET_FRACS = [0.01, 0.02, 0.05, 0.10, 0.20]     # inspection budget k as a share of the deployment window
N_SPLITS = 200               # repeated calibration/deployment splits for the validity check
MIN_ATTACKS_FOR_CLASS = 10   # a per-class certificate is reported only above this many calibration attacks

# Baselines at a fixed RISK level, to show their selected-set size is an output and violates a hard budget
RISK_LEVELS = [0.05, 0.10, 0.20]
FDR_LEVELS = [0.05, 0.10, 0.20]

TABLES = Path(REPO) / 'results' / 'tables'
FIGS = Path(REPO) / 'results' / 'figures'
PREFIX = 'budget_recall'

def find_proba_file(dataset, model_name, split):
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / f'{model_name}_{split}_proba.npy'
        if p.exists():
            return p
    raise FileNotFoundError(f'{dataset}/{model_name}_{split}_proba.npy')
print('ready')


In [ ]:
def cp_lower(k_succ, n, d):
    """Clopper-Pearson lower confidence bound on a binomial rate. Exact, distribution-free, finite-sample."""
    if n == 0 or k_succ == 0:
        return 0.0
    return float(BetaDist.ppf(d, k_succ, n - k_succ + 1))

def future_frac_lower(n_future, p, d):
    """Lower bound, holding with probability at least 1 - d, on Binomial(n_future, p) / n_future.
    This is the term the obvious construction omits. It is what makes the certificate valid for rare classes."""
    if n_future == 0:
        return 0.0
    return float(binom.ppf(d, n_future, p)) / n_future

def cp_upper(k_succ, n, d):
    """Clopper-Pearson upper confidence bound on a binomial rate."""
    if n == 0:
        return 1.0
    if k_succ >= n:
        return 1.0
    return float(BetaDist.ppf(1 - d, k_succ + 1, n - k_succ))

def count_upper(n_future, p, d):
    """Upper bound, holding with probability at least 1 - d, on Binomial(n_future, p)."""
    return float(binom.ppf(1 - d, n_future, min(max(p, 0.0), 1.0)))

def budget_safe_threshold(s_cal, n_dep, k, d):
    """Largest calibration rank whose deployment count is still within the budget k with high probability.

    Fixing the inspection COUNT rather than the risk level introduces a random quantity that fixed-risk
    methods never face: the number of deployment alerts above the threshold. If that count exceeds k, the
    operator cannot inspect them all, and a bound written for the threshold describes a set the operator
    never sees. Choosing the rank so the count stays inside the budget makes everything above the threshold
    a subset of the inspected top-k, which is what licenses the recall bound.
    """
    n_cal = len(s_cal)
    srt = np.sort(s_cal)[::-1]
    lo, hi, best = 1, n_cal, 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if count_upper(n_dep, cp_upper(mid, n_cal, d), d) <= k:
            best = mid; lo = mid + 1
        else:
            hi = mid - 1
    return float(srt[best - 1]), best

def certify_recall_at_k(s_cal, a_cal, s_dep, k, delta=DELTA, n_attacks_dep=None, mode='three_stage'):
    """Certificate: with probability at least 1 - delta, the fraction of attacks in the deployment window
    that fall inside the k highest-scoring alerts is at least the returned bound.

    Recall at a FIXED count has three sources of randomness, and the error budget is split across all three:
      stage 1  the number of deployment alerts above the threshold, bounded so the certified set fits inside
               the operator's budget (omitted by the two-stage and calibration-only variants);
      stage 2  the calibration split estimates q = P(score >= tau | attack) with error, bounded exactly;
      stage 3  the deployment window then realises a finite number of attacks whose caught fraction
               fluctuates around q, bounded at the realised attack count.
    Stage 3 dominates when attacks are rare, which is why the calibration-only variant under-covers precisely
    on the classes an intrusion detector most needs to catch. The variants are kept as ablations.
    """
    n_cal = len(s_cal)
    if n_cal == 0 or a_cal.sum() == 0:
        return dict(bound=0.0, tau=np.nan, q_lower=0.0, n_attacks_cal=int(a_cal.sum()), cal_rank=0)
    n_dep = max(len(s_dep), 1)
    if mode == 'three_stage':
        d = delta / 3.0
        tau, rank = budget_safe_threshold(s_cal, n_dep, k, d)
        d_cal, d_fut = d, d
    else:
        d_cal = delta * SPLIT
        d_fut = delta * (1 - SPLIT)
        rank = min(max(1, int(round(k / n_dep * n_cal))), n_cal)
        tau = float(np.sort(s_cal)[::-1][rank - 1])
    hits = int(((s_cal >= tau) & (a_cal == 1)).sum())
    n_att_cal = int(a_cal.sum())
    q_lo = cp_lower(hits, n_att_cal, d_cal)
    if mode == 'calibration_only' or n_attacks_dep is None:
        b = q_lo
    else:
        b = future_frac_lower(int(n_attacks_dep), q_lo, d_fut)
    return dict(bound=b, tau=float(tau), q_lower=q_lo, n_attacks_cal=n_att_cal, cal_rank=int(rank))

def realised_recall_at_k(s_dep, a_dep, k):
    if a_dep.sum() == 0:
        return np.nan
    order = np.argsort(-s_dep)[:k]
    return float(a_dep[order].sum() / a_dep.sum())

def realised_recall_at_tau(s_dep, a_dep, tau):
    if a_dep.sum() == 0:
        return np.nan, 0
    sel = s_dep >= tau
    return float(a_dep[sel].sum() / a_dep.sum()), int(sel.sum())
print('certificate ready')


In [ ]:
def crc_threshold_for_risk(s_cal, a_cal, alpha, delta=DELTA):
    """Conformal-risk-control style baseline: fix a RISK level (here, missed-attack rate among attacks) and
    search for the smallest threshold meeting it on the calibration split. The number of alerts this selects
    is an OUTPUT, not an input, which is precisely what a hard analyst budget forbids."""
    order = np.argsort(-s_cal)
    a_sorted = a_cal[order]; s_sorted = s_cal[order]
    n_att = max(int(a_cal.sum()), 1)
    caught = np.cumsum(a_sorted)
    miss = 1.0 - caught / n_att
    ok = np.where(miss <= alpha)[0]
    if len(ok) == 0:
        return float(s_sorted[-1]), len(s_cal)
    j = ok.min()
    return float(s_sorted[j]), int(j + 1)

def conformal_selection_bh(s_cal, a_cal, s_dep, fdr):
    """Jin and Candes style conformal selection: conformal p-values against the null 'not an attack',
    then Benjamini-Hochberg. Controls the FALSE DISCOVERY side. The selected-set size floats and no recall
    lower bound is produced, which is the distinction this paper turns on."""
    null_scores = s_cal[a_cal == 0]
    n0 = len(null_scores)
    if n0 == 0:
        return np.zeros(len(s_dep), dtype=bool), np.nan
    p = (1.0 + np.searchsorted(np.sort(null_scores), s_dep, side='left')) / (n0 + 1.0)
    p = 1.0 - p + 1.0 / (n0 + 1.0)      # small p for attack-like scores
    m = len(p); order = np.argsort(p); ranked = p[order]
    thresh = ranked <= (np.arange(1, m + 1) / m) * fdr
    if not thresh.any():
        return np.zeros(m, dtype=bool), 0
    cut = np.where(thresh)[0].max()
    sel = np.zeros(m, dtype=bool); sel[order[:cut + 1]] = True
    return sel, int(sel.sum())

def dual_threshold_deferral(s_cal, a_cal, s_dep, beta):
    """Dual-threshold deferral baseline: choose a low threshold so that the probability an auto-closed alert
    is a real attack is at most beta. Fixes an error PROBABILITY, not a count."""
    order = np.argsort(s_cal)
    a_sorted = a_cal[order]; s_sorted = s_cal[order]
    n_att = max(int(a_cal.sum()), 1)
    missed = np.cumsum(a_sorted) / n_att
    ok = np.where(missed <= beta)[0]
    tau = float(s_sorted[ok.max()]) if len(ok) else float(s_sorted[0])
    return tau, int((s_dep >= tau).sum())
print('baselines ready')


In [ ]:
def fit_one_map(x, y):
    if len(x) < 2 or len(np.unique(y)) < 2:
        c = float(y.mean()) if len(y) else 0.5
        return lambda p, c=c: np.full(len(p), c)
    if len(x) >= PLATT_THRESHOLD:
        iso = IsotonicRegression(out_of_bounds='clip').fit(x, y)
        return lambda p, iso=iso: iso.predict(p)
    lr = LogisticRegression(C=1e10, solver='lbfgs').fit(x.reshape(-1, 1), y)
    return lambda p, lr=lr: lr.predict_proba(p.reshape(-1, 1))[:, 1]

def build_scores(P_fit, y_fit, P_apply):
    """Alert scores. The triage score is the calibrated probability that the flow is an attack of any kind,
    calibrated on the fitting slice. A rare-class-weighted variant upweights the classes an operator most
    wants to catch. Both are ranking scores only; no decision is changed."""
    a_fit = np.isin(y_fit, ATTACK_CLASSES).astype(float)
    raw_fit = 1.0 - P_fit[:, 0]; raw_app = 1.0 - P_apply[:, 0]
    m = fit_one_map(raw_fit, a_fit)
    plain = m(raw_app)
    # Isotonic calibration maps whole blocks of alerts to one value. An operator cannot inspect part of a tie
    # block, so a fixed-budget rule is not even well defined until ties are broken. Breaking them by the raw
    # score at a scale of 1e-6 changes no calibrated value materially and no bin assignment.
    def tb(cal, raw):
        r = float(raw.max() - raw.min())
        return cal * (1 - 1e-6) + 1e-6 * ((raw - raw.min()) / (r + 1e-12))
    plain = tb(plain, raw_app)
    prev = np.array([max((y_fit == c).mean(), 1e-6) for c in range(K)])
    w = np.zeros(K); w[ATTACK_CLASSES] = 1.0 / prev[ATTACK_CLASSES]
    w = w / w.max()
    weighted = (P_apply * w[None, :]).sum(1)
    return {'calibrated_attack_prob': plain, 'rare_weighted': weighted, 'uncalibrated': raw_app}
print('scores ready')


In [ ]:
t0 = time.time()
rows, base_rows = [], []
rng_global = np.random.RandomState(SEED)
for ds in DATASETS:
    proc = Path(REPO) / 'data' / 'processed' / ds
    y_cal_full = np.load(proc / 'y_calib_5class.npy'); y_te = np.load(proc / 'y_test_5class.npy')
    a_te = np.isin(y_te, ATTACK_CLASSES).astype(int)
    print(f'\n=== {DS_LABEL[ds]} ({EXCHANGEABLE[ds]}): calib={len(y_cal_full)} test={len(y_te)} '
          f'test attack rate={a_te.mean():.3f} test class counts={np.bincount(y_te, minlength=K).tolist()} ===')
    for model in MODELS:
        P_cal_full = np.load(find_proba_file(ds, model, 'calib')).astype(np.float64)
        P_te = np.load(find_proba_file(ds, model, 'test')).astype(np.float64)
        P_cal_full /= np.maximum(P_cal_full.sum(1, keepdims=True), EPS); P_te /= np.maximum(P_te.sum(1, keepdims=True), EPS)
        for rep in range(N_SPLITS):
            rs = np.random.RandomState(1000 * rep + 17)
            # the calibration partition is split into a slice that fits the score and a slice that certifies;
            # the deployment window is a random half of the test partition
            cperm = rs.permutation(len(y_cal_full))
            fit_idx, cert_idx = cperm[:len(cperm) // 2], cperm[len(cperm) // 2:]
            dperm = rs.permutation(len(y_te)); dep_idx = dperm[:len(dperm) // 2]
            y_fit, y_cert = y_cal_full[fit_idx], y_cal_full[cert_idx]
            sc = build_scores(P_cal_full[fit_idx], y_fit, P_cal_full[cert_idx])
            sd = build_scores(P_cal_full[fit_idx], y_fit, P_te[dep_idx])
            a_cert = np.isin(y_cert, ATTACK_CLASSES).astype(int)
            a_dep = a_te[dep_idx]; y_dep = y_te[dep_idx]
            n_dep = len(dep_idx)
            for score_name in sc:
                s_cert, s_dep = sc[score_name], sd[score_name]
                for frac in BUDGET_FRACS:
                    k = max(1, int(round(frac * n_dep)))
                    # marginal certificate over all attacks
                    cert = certify_recall_at_k(s_cert, a_cert, s_dep, k, n_attacks_dep=int(a_dep.sum()), mode='three_stage')
                    two = certify_recall_at_k(s_cert, a_cert, s_dep, k, n_attacks_dep=int(a_dep.sum()), mode='two_stage')
                    naive = certify_recall_at_k(s_cert, a_cert, s_dep, k, mode='calibration_only')
                    emp_k = realised_recall_at_k(s_dep, a_dep, k)
                    emp_tau, n_sel = realised_recall_at_tau(s_dep, a_dep, cert['tau'])
                    rows.append({'dataset': ds, 'exchangeability': EXCHANGEABLE[ds], 'model': model, 'rep': rep,
                                 'score': score_name, 'budget_frac': frac, 'k': k, 'scope': 'all_attacks',
                                 'bound_three_stage': cert['bound'], 'bound_two_stage': two['bound'], 'bound_calibration_only': naive['bound'],
                                 'q_lower': cert['q_lower'], 'tau': cert['tau'], 'cal_rank': cert['cal_rank'],
                                 'n_attacks_cal': cert['n_attacks_cal'], 'n_attacks_dep': int(a_dep.sum()),
                                 'empirical_recall_at_k': emp_k, 'empirical_recall_at_tau': emp_tau,
                                 'n_selected_at_tau': n_sel})
                    # per-class certificates: marginal recall is dominated by the common attack classes,
                    # so a rare-class floor is the quantity an operator actually needs
                    for c in ATTACK_CLASSES:
                        ac_cert = (y_cert == c).astype(int); ac_dep = (y_dep == c).astype(int)
                        if ac_cert.sum() < MIN_ATTACKS_FOR_CLASS or ac_dep.sum() == 0:
                            continue
                        cc = certify_recall_at_k(s_cert, ac_cert, s_dep, k, n_attacks_dep=int(ac_dep.sum()), mode='three_stage')
                        tt = certify_recall_at_k(s_cert, ac_cert, s_dep, k, n_attacks_dep=int(ac_dep.sum()), mode='two_stage')
                        nn = certify_recall_at_k(s_cert, ac_cert, s_dep, k, mode='calibration_only')
                        rows.append({'dataset': ds, 'exchangeability': EXCHANGEABLE[ds], 'model': model, 'rep': rep,
                                     'score': score_name, 'budget_frac': frac, 'k': k, 'scope': CLASS_NAMES[c],
                                     'bound_three_stage': cc['bound'], 'bound_two_stage': tt['bound'], 'bound_calibration_only': nn['bound'],
                                     'q_lower': cc['q_lower'], 'tau': cc['tau'], 'cal_rank': cc['cal_rank'],
                                     'n_attacks_cal': cc['n_attacks_cal'], 'n_attacks_dep': int(ac_dep.sum()),
                                     'empirical_recall_at_k': realised_recall_at_k(s_dep, ac_dep, k),
                                     'empirical_recall_at_tau': realised_recall_at_tau(s_dep, ac_dep, cc['tau'])[0],
                                     'n_selected_at_tau': int((s_dep >= cc['tau']).sum())})
            # baselines: what set size does a fixed-RISK or fixed-FDR rule actually select?
            if rep < 20:
                s_cert, s_dep = sc['calibrated_attack_prob'], sd['calibrated_attack_prob']
                for alpha in RISK_LEVELS:
                    tau_c, n_cal_sel = crc_threshold_for_risk(s_cert, a_cert, alpha)
                    rec, nsel = realised_recall_at_tau(s_dep, a_dep, tau_c)
                    base_rows.append({'dataset': ds, 'model': model, 'rep': rep, 'method': 'fixed_risk_crc',
                                      'level': alpha, 'selected_frac_of_window': nsel / n_dep,
                                      'realised_recall': rec, 'has_recall_lower_bound': False})
                    tau_d, nsel_d = dual_threshold_deferral(s_cert, a_cert, s_dep, alpha)
                    rec_d, _ = realised_recall_at_tau(s_dep, a_dep, tau_d)
                    base_rows.append({'dataset': ds, 'model': model, 'rep': rep, 'method': 'dual_threshold_deferral',
                                      'level': alpha, 'selected_frac_of_window': nsel_d / n_dep,
                                      'realised_recall': rec_d, 'has_recall_lower_bound': False})
                for f in FDR_LEVELS:
                    sel, nsel = conformal_selection_bh(s_cert, a_cert, s_dep, f)
                    rec = float(a_dep[sel].sum() / max(a_dep.sum(), 1)) if nsel else 0.0
                    base_rows.append({'dataset': ds, 'model': model, 'rep': rep, 'method': 'conformal_selection_fdr',
                                      'level': f, 'selected_frac_of_window': nsel / n_dep,
                                      'realised_recall': rec, 'has_recall_lower_bound': False})
        print(f'  {model:18s} done  {(time.time() - t0) / 60:.1f} min')
df = pd.DataFrame(rows); df.to_csv(TABLES / f'{PREFIX}_certificates.csv', index=False)
df_base = pd.DataFrame(base_rows); df_base.to_csv(TABLES / f'{PREFIX}_baselines.csv', index=False)
print(f'\n{len(df)} certificate rows, {len(df_base)} baseline rows; {(time.time() - t0) / 60:.1f} min')


In [ ]:
pd.set_option('display.width', 250)
for v in ['three_stage', 'two_stage', 'calibration_only']:
    df[f'violated_{v}'] = df.empirical_recall_at_k < df[f'bound_{v}']
df['tightness'] = df.bound_three_stage / df.empirical_recall_at_k.replace(0, np.nan)
df['budget_overrun'] = df.n_selected_at_tau / df.k
main = df[df.score == 'calibrated_attack_prob']

print('=== 1. VALIDITY: violation rate against the nominal %.2f, by dataset and scope ===' % DELTA)
v = main.groupby(['exchangeability', 'dataset', 'scope']).agg(
    n=('violated_three_stage', 'size'), viol_three_stage=('violated_three_stage', 'mean'),
    viol_two_stage=('violated_two_stage', 'mean'), viol_calibration_only=('violated_calibration_only', 'mean'),
    median_budget_overrun=('budget_overrun', 'median'),
    mean_bound=('bound_three_stage', 'mean'), mean_empirical=('empirical_recall_at_k', 'mean'),
    median_attacks_in_window=('n_attacks_dep', 'median')).reset_index()
print(v.round(4).to_string(index=False))

print('\n=== 2. ABLATION: each dropped stage costs validity (violation rate against the nominal %.2f) ===' % DELTA)
print(main.groupby('scope').agg(viol_three_stage=('violated_three_stage', 'mean'),
      viol_two_stage=('violated_two_stage', 'mean'), viol_calibration_only=('violated_calibration_only', 'mean'),
      median_attacks=('n_attacks_dep', 'median'), median_budget_overrun=('budget_overrun', 'median')).round(4).to_string())

print('\n=== 3. NON-VACUITY: certified floor against the naive top-k it would replace ===')
nv = main.groupby(['dataset', 'scope', 'budget_frac']).agg(
    mean_bound=('bound_three_stage', 'mean'), mean_empirical=('empirical_recall_at_k', 'mean'),
    tightness=('tightness', 'median'), frac_bound_above_zero=('bound_three_stage', lambda s: float((s > 0).mean()))).reset_index()
print(nv[nv.budget_frac.isin([0.02, 0.05, 0.10])].round(3).to_string(index=False))

print('\n=== 4. BASELINES: a fixed risk or FDR level does not respect a hard inspection budget ===')
print(df_base.groupby(['dataset', 'method', 'level']).agg(
    median_selected_frac_of_window=('selected_frac_of_window', 'median'),
    p90_selected_frac=('selected_frac_of_window', lambda s: float(np.percentile(s, 90))),
    median_recall=('realised_recall', 'median')).round(3).to_string())

print('\n=== 5. score choice ===')
print(df[df.scope == 'all_attacks'].groupby(['dataset', 'score']).agg(
    mean_bound=('bound_three_stage', 'mean'), mean_empirical=('empirical_recall_at_k', 'mean'),
    viol=('violated_three_stage', 'mean')).round(3).to_string())

cic = main[main.dataset == 'cic_ids2017_v2']
rare = main[main.scope.isin(['R2L', 'U2R'])]
op = main[main.budget_frac.isin([0.02, 0.05])]
crit = {'timestamp': datetime.now().isoformat(), 'notebook': '18_budget_recall_certificate.ipynb',
        'delta': DELTA, 'delta_split_two_stage': SPLIT, 'delta_split_three_stage': 'equal thirds', 'n_splits': N_SPLITS, 'budget_fracs': BUDGET_FRACS,
        'KILL_1_validity_on_exchangeable': {
            'rule': 'fails if the two-stage violation rate on CIC-IDS2017 exceeds the nominal delta',
            'violation_rate_cic': float(cic.violated_two_stage.mean()), 'nominal': DELTA,
            'violation_rate_cic_by_scope': cic.groupby('scope').violated_two_stage.mean().round(4).to_dict()},
        'KILL_2_non_vacuity': {
            'rule': 'fails if the certified floor is at or below zero at operational budgets on the exchangeable dataset',
            'frac_bound_above_zero_cic_operational': float((cic[cic.budget_frac.isin([0.02, 0.05])].bound_three_stage > 0).mean()),
            'median_tightness_cic': float(cic.tightness.median()),
            'mean_bound_cic_by_scope': cic.groupby('scope').bound_three_stage.mean().round(4).to_dict()},
        'KILL_3_rare_class_floor': {
            'rule': 'weak if the rare-class floor is vacuous everywhere',
            'frac_above_zero': float((rare.bound_three_stage > 0).mean()),
            'mean_bound_by_dataset': rare.groupby('dataset').bound_three_stage.mean().round(4).to_dict(),
            'violation_rate': float(rare.violated_two_stage.mean())},
        'STAGE2_NECESSITY': {
            'rule': 'the two-stage construction is only justified if the calibration-only bound under-covers',
            'viol_calibration_only_by_scope': main.groupby('scope').violated_calibration_only.mean().round(4).to_dict(),
            'viol_two_stage_by_scope': main.groupby('scope').violated_two_stage.mean().round(4).to_dict()},
        'BUDGET_VIOLATION_BY_BASELINES': {
            'rule': 'fixed-risk and fixed-FDR methods select a set whose size is an output, not an input',
            'median_selected_frac': df_base.groupby('method').selected_frac_of_window.median().round(3).to_dict(),
            'p90_selected_frac': df_base.groupby('method').selected_frac_of_window.apply(
                lambda s: float(np.percentile(s, 90))).round(3).to_dict()},
        'SHIFT_LADDER': {'violation_rate_by_exchangeability':
            main.groupby('exchangeability').violated_two_stage.mean().round(4).to_dict()}}
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump(crit, f, indent=2, default=float)
print('\n=== KILL CRITERIA ===')
for k_ in crit:
    if k_.startswith(('KILL', 'STAGE', 'BUDGET', 'SHIFT')):
        print(json.dumps({k_: crit[k_]}, indent=1, default=float))


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"
import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '18_budget_recall_certificate.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')
!git add notebooks/18_budget_recall_certificate.ipynb results/tables/budget_recall_*.csv results/tables/budget_recall_summary.json
!git status --short | head -20
!git commit -m "Notebook 18: distribution-free recall certificate at a fixed inspection budget. Three-stage bound over the selected count, the calibration estimate and the realised attack count, with tie-broken scores, marginal and per attack class, against fixed-risk CRC, conformal selection under FDR control and dual-threshold deferral, across the three datasets ordered by partition exchangeability"
!git push origin main
!git log --oneline -2
